In [3]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import pandas as pd
from tqdm import tqdm

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando {device}")

Usando cpu


In [5]:
# 1. DATOS DE EJEMPLO (comentarios en español)
comentarios = [
    # Positivos (1)
    "Me encantó este producto, es excelente",
    "Muy buen servicio, lo recomiendo totalmente",
    "La atención al cliente fue increíble",
    "Excelente calidad, superó mis expectativas",
    "Me siento muy satisfecho con mi compra",
    "El producto llegó antes de lo esperado",
    "Totalmente recomendado, funciona perfecto",
    "La mejor compra que he hecho este año",
    "El personal es muy amable y profesional",
    "Volvería a comprar sin dudarlo",
    
    # Negativos (0)
    "Pésimo servicio, no lo recomiendo",
    "El producto llegó roto y en mal estado",
    "Muy mala experiencia, no volveré a comprar",
    "La atención al cliente es terrible",
    "No funciona como esperaba, decepcionado",
    "El envío tardó mucho más de lo debido",
    "Producto de baja calidad, no vale lo que cuesta",
    "Me arrepiento de haber comprado esto",
    "El servicio al cliente no responde",
    "No cumple con lo prometido, muy malo"
]


In [7]:
etiquetas = [1]*10 + [0]*10
etiquetas

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

In [18]:
df = pd.DataFrame({'comentario':comentarios, 'sentimiento':etiquetas})
df

,comentario,sentimiento
0,"Me encantó este producto, es excelente",1
1,"Muy buen servicio, lo recomiendo totalmente",1
2,La atención al cliente fue increíble,1
3,"Excelente calidad, superó mis expectativas",1
4,Me siento muy satisfecho con mi compra,1
5,El producto llegó antes de lo esperado,1
6,"Totalmente recomendado, funciona perfecto",1
7,La mejor compra que he hecho este año,1
8,El personal es muy amable y profesional,1
9,Volvería a comprar sin dudarlo,1


# DataSet personalizado para PyTorch

In [10]:
class SentimentDataset(Dataset):
    def __init__(self, textos, etiquetas, tokenizaer, max_len = 128):
        self.textos = textos
        self.etiqueteas = etiquetas
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.textos)

    def _getitem_(self, idx):
        texto = str(self.textos[idx])
        etiqueta = self.etiqeutas[idx]

        enconding = self.tokeinzer(
            texto,
            max_lenght = self.max_len,
            padding = 'max_length',
            truncation = True,
            return_tensors = 'pt'
        )

        return {
            'inputs_ids':encoding['inputs_ids'].flatten(),
            'attention_mask':encoding['attention_mask'].flatten(),
            'labels':torch.tensor(etiqueta, dtype = torch.long)
        }

In [13]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

C:\Users\Danie\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Danie\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [20]:
x = np.array(comentarios)
y = np.array(etiquetas, dtype=n.int64)

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size = 0.3,
    random_state = 23,
    stratify = df['sentimiento'].values
)

array(['Muy mala experiencia, no volveré a comprar',
       'El producto llegó roto y en mal estado',
       'La atención al cliente fue increíble',
       'Muy buen servicio, lo recomiendo totalmente',
       'Me encantó este producto, es excelente',
       'No funciona como esperaba, decepcionado',
       'La mejor compra que he hecho este año',
       'El servicio al cliente no responde',
       'No cumple con lo prometido, muy malo',
       'El personal es muy amable y profesional',
       'El producto llegó antes de lo esperado',
       'Me arrepiento de haber comprado esto',
       'Me siento muy satisfecho con mi compra',
       'El envío tardó mucho más de lo debido'], dtype=object)

In [23]:
train_dataset = SentimentDataset(x_train, y_train, tokenizer)
test_dataset = SentimentDataset(x_test, y_test, tokenizer)


In [24]:
train_dataloader = DataLoader(train_dataset, batch_size = 4, shuffle = True)
test_dataloader = DataLoader(test_dataset, batch_size = 4, shuffle = True)

## Entrenamiento del modelo

In [25]:
modelo = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels = 2
)
modelo = modelo.to(device)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [26]:
optimizador = AdamW(modelo.parameters(), lr=2e-5)

In [ ]:
def evaluar_modelo():

In [2]:
def entrenar_modelo(model, train_loader, test_loader, optimizer, epocas = 5):
    train_losses = []
    test_accuracies = []
    for epoca in range(epocas):
        print(f'Epoca {epoca+1}/{epocas}:')
        print('-'*50)
        # Modo entrenamiento
        model.train()
        total_loss = 0
        num_batches = 0

        for batch_idx, batch in enumerate(tqdm(train_loader, desc = 'Entrenanando')):
            # mover datos al dispositivo
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            ## pasar
            optimizer.zero_grad()
            outputs = model(
                inputs_ids = inputs_ids,
                attention_mask = attention_mask,
                labels = labels
            )
            loss = outpus.loss
            total_loss += loss
            num_batches += 1

            # pasar hacia atrás

            loss.backward()
            optimizer.step()
        avg_loss = total_loss/num_batches if num_batches>0 else 0
        train_losses.append(avg_leos)

        # Evaluacion
        accuracy = evaluar_modelo(model, test_loader)
        test_accuracies.append(accuracy)
        print(f'Promedio de perdidas: {avg_loss:.4f}')
        print(f'Promedio de pruebas: {accuracy:.4f}')
    return train_losses, test_accuracies